In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

from langchain.messages import HumanMessage
from langchain_deepseek import ChatDeepSeek

from dotenv import load_dotenv
load_dotenv(override=True)

model = ChatDeepSeek(
    model="deepseek-v4-flash",
    extra_body={
        "thinking": {
            "type": "disabled"
        }
    }
)

# グラフの状態
class OverAllState(TypedDict):
    topic: str
    joke: str
    story: str
    poem: str
    combined_output: str

# ノード
def call_model_1(state: OverAllState) -> OverAllState:
    """1回目の大規模言語モデル呼び出し。ジョークを生成する"""

    msg = model.invoke(
        [HumanMessage(content=f"「{state['topic']}」についての短いジョークを書いてください")]
    )
    return {"joke": msg.content}

def call_model_2(state: OverAllState) -> OverAllState:
    """2回目の大規模言語モデル呼び出し。物語を生成する"""

    msg = model.invoke(
        [HumanMessage(content=f"「{state['topic']}」についての短い物語を書いてください")]
    )
    return {"story": msg.content}

def call_model_3(state: OverAllState) -> OverAllState:
    """3回目の大規模言語モデル呼び出し。詩を生成する"""

    msg = model.invoke(
        [HumanMessage(content=f"「{state['topic']}」についての短い詩を書いてください")]
    )
    return {"poem": msg.content}

def aggregator(state: OverAllState) -> OverAllState:
    """ジョーク、物語、詩を1つの出力にまとめる"""

    combined = f"以下は「{state['topic']}」についての物語、ジョーク、詩です！\n\n"
    combined += f"物語：\n{state['story']}\n\n"
    combined += f"ジョーク：\n{state['joke']}\n\n"
    combined += f"詩：\n{state['poem']}"
    return {"combined_output": combined}

# ワークフローを構築
builder = StateGraph(state_schema=OverAllState)

# ノードを追加
builder.add_node("call_model_1", call_model_1)
builder.add_node("call_model_2", call_model_2)
builder.add_node("call_model_3", call_model_3)
builder.add_node("aggregator", aggregator)

# エッジを追加して各ノードを接続
builder.add_edge(START, "call_model_1")
builder.add_edge(START, "call_model_2")
builder.add_edge(START, "call_model_3")
builder.add_edge(["call_model_1", "call_model_2", "call_model_3"], "aggregator")
builder.add_edge("aggregator", END)

graph = builder.compile()

# ワークフローを呼び出す
response = graph.invoke({"topic": "猫"})
print(response)

from IPython.display import display
display(graph)